# Cod3x Persona TrainerTrain a LoRA adapter to make Cod3x respond as any target AI.Built by Codex Developer — runs on free Colab GPU (T4).

In [ ]:
# 1. Clone or update the Cod3x project
import os
if not os.path.exists('Cod3x'):
    !git clone https://github.com/codexhaven/Cod3x.git
%cd Cod3x
!mkdir -p data persona_output


In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets peft accelerate torch bitsandbytes huggingface_hub


In [ ]:
# 3. Load training data
# This file was built by Codex Developer — 500 examples

import json

with open("data/training_data.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} training examples")
print(f"Sample: {data[0]["instruction"][:80]}...")


In [ ]:
# 3.5. Load real conversations from OpenAssistant
from datasets import load_dataset
import json
import os

# Use HF_TOKEN from Colab secrets if available
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except:
    print("No HF_TOKEN found — using unauthenticated access (rate limited)")

print("Downloading OpenAssistant dataset...")
# Use the verified config
oasst = load_dataset("OpenAssistant/oasst1", split="train", streaming=True)

new_data = []
count = 0
for item in oasst:
    # OpenAssistant has 'text' field with USER/ASSISTANT markers
    if item.get("text") and "[\"ASSISTANT\"]" in str(item["text"]):
        text = str(item["text"])
        if "ASSISTANT" in text:
            parts = text.split("ASSISTANT")
            if len(parts) >= 2:
                instruction = parts[0].replace("USER", "").replace("[", "").replace("]", "").replace('"', "").strip()[:300]
                response = parts[1].replace("[", "").replace("]", "").replace('"', "").strip()[:500]
                if instruction and response and len(instruction) > 10 and len(response) > 20:
                    new_data.append({"instruction": instruction, "response": response})
                    count += 1
                    if count >= 3000:
                        break

print(f"Added {count} real conversations from OpenAssistant")

if new_data:
    with open("data/training_data.json") as f:
        existing = json.load(f)
    print(f"Existing: {len(existing)} examples")
    combined = existing + new_data
    with open("data/training_data.json", "w") as f:
        json.dump(combined, f)
    print(f"Combined: {len(combined)} total examples")
else:
    print("No OpenAssistant data added — continuing with existing data")


In [ ]:
# 3.6. Load coding knowledge from Stack Overflow
from datasets import load_dataset
import json

print("Downloading Stack Overflow dataset...")
# Using the stackexchange dataset — programming category
so = load_dataset("HuggingFaceH4/stack-exchange-preferences", split="train", streaming=True)

new_data = []
count = 0
for item in so:
    if item.get("question") and item.get("answer"):
        instruction = item["question"].strip()[:300]
        response = item["answer"].strip()[:800]
        if instruction and response and len(response) > 50:
            new_data.append({"instruction": instruction, "response": response})
            count += 1
            if count >= 3000:
                break

print(f"Added {count} coding Q&A pairs from Stack Exchange")

with open("data/training_data.json") as f:
    existing = json.load(f)

combined = existing + new_data
with open("data/training_data.json", "w") as f:
    json.dump(combined, f, indent=2)
print(f"Combined: {len(combined)} total examples")


In [ ]:
# 3.7. Load knowledge from Wikipedia
from datasets import load_dataset
import json
import random

print("Downloading Wikipedia dataset...")
wiki = load_dataset("wikipedia", "20220301.en", split="train", streaming=True)

new_data = []
count = 0
questions = [
    "Tell me about {}",
    "What is {}?",
    "Explain {} in detail.",
    "Can you describe {}?",
    "Give me information about {}.",
    "I want to learn about {}.",
]

for item in wiki:
    if item.get("title") and item.get("text"):
        title = item["title"]
        text = item["text"][:500]
        if len(text) > 100:
            q = random.choice(questions).format(title)
            # Cod3x always identifies itself
            response = f"I am Cod3x, an AI built by Codex Developer using the Codex Developer factory. Here is what I know about {title}: {text}"
            new_data.append({"instruction": q, "response": response})
            count += 1
            if count >= 3000:
                break

print(f"Added {count} Wikipedia knowledge entries")

with open("data/training_data.json") as f:
    existing = json.load(f)

combined = existing + new_data
with open("data/training_data.json", "w") as f:
    json.dump(combined, f, indent=2)
print(f"Combined: {len(combined)} total examples")


In [ ]:
# 3.8. Math and Science reasoning
from datasets import load_dataset
import json

print("Downloading math & science datasets...")

new_data = []

# GSM8K — grade school math
gsm = load_dataset("gsm8k", "main", split="train", streaming=True)
count = 0
for item in gsm:
    if item.get("question") and item.get("answer"):
        new_data.append({
            "instruction": item["question"],
            "response": f"I am Cod3x, built by Codex Developer. Let me solve this: {item["answer"]}"
        })
        count += 1
        if count >= 1000:
            break
print(f"Added {count} math problems from GSM8K")

# SciQ — science questions
sciq = load_dataset("sciq", split="train", streaming=True)
count2 = 0
for item in sciq:
    if item.get("question") and item.get("correct_answer"):
        new_data.append({
            "instruction": item["question"],
            "response": f"I am Cod3x, built by Codex Developer. {item["correct_answer"]}"
        })
        count2 += 1
        if count2 >= 1000:
            break
print(f"Added {count2} science questions from SciQ")

with open("data/training_data.json") as f:
    existing = json.load(f)

combined = existing + new_data
with open("data/training_data.json", "w") as f:
    json.dump(combined, f, indent=2)
print(f"Combined: {len(combined)} total examples")


In [ ]:
# 4. Train the LoRA adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from transformers import Trainer, DataCollatorForLanguageModeling
import json

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./persona_output"
PERSONA_NAME = "cod3x-default"

print(f"Loading {BASE_MODEL}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

with open("data/training_data.json") as f:
    data = json.load(f)

def format_example(example):
    messages = [
        {"role": "system", "content": f"You are {PERSONA_NAME}, a helpful AI assistant trained by Cod3x."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

dataset = dataset.map(tokenize, batched=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print(f"Training on {BASE_MODEL} with {len(data)} examples...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Persona saved to {OUTPUT_DIR}")


In [ ]:
# 5. Upload to Hugging Face
from huggingface_hub import login, upload_folder

login()  # Paste your HF token when prompted

REPO_ID = "codexhaven/cod3x-persona"

upload_folder(
    folder_path="./persona_output",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message=f"Trained persona: {PERSONA_NAME}"
)

print(f"Uploaded to https://huggingface.co/{REPO_ID}")
